# ACP et UMAP sur le vin rouge (Wine Quality)

Le jeu Wine Quality (Cortez et al., 2009) décrit 1 599 vins rouges portugais (vinho verde) par 11 mesures physico-chimiques (acidité, sucre, chlorures, soufre, densité, pH, sulfates, alcool) et une note de qualité donnée par des dégustateurs, de 3 à 8. Sur ce jeu réel, vous réduisez la dimension et vous interprétez le résultat :

1. Faut-il réduire ?
2. Choisir k
3. Lire les axes : les vins extrêmes
4. La qualité se lit-elle sur la projection ?
5. Projections non linéaires : UMAP et t-SNE

La cellule de départ charge les données dans `df` (la note dans la colonne `quality`), met les 11 mesures dans `X` et leur version centrée réduite dans `X_scaled`. Les parties 1 et 2 sont indépendantes ; les parties 3 à 5 s'enchaînent (la 3 calcule la projection `X_pca`, la 4 les groupes de qualité que la 5 réutilise). Pas besoin de GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh wine-quality

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# La première colonne du CSV numérote les vins : c'est l'index, pas une mesure
df = pd.read_csv("wine-quality/wineQualityReds.csv", index_col=0)
print(df.shape)
display(df.head(10))

X = df.drop(columns="quality")
X_scaled = StandardScaler().fit_transform(X)
print("X :", X.shape, "X_scaled :", X_scaled.shape)

## 1. Faut-il réduire ?

L'ACP de scikit-learn centre les données sans les réduire. Ajustez une `PCA` qui garde tous les axes sur `X`, puis une autre sur `X_scaled`. Pour chacune, affichez la part de variance de la première composante, PC1, et ses plus forts coefficients en valeur absolue (`components_[0]`, avec le nom de chaque mesure). Affichez aussi l'écart-type de chaque mesure de `X`.

- Sans réduction, quelle mesure fait PC1 ? Pourquoi celle-là ?
- Faut-il réduire sur ce jeu ? Que fait PC1 une fois les mesures réduites ?

**Documentation utile**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`pandas.Series.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html)
- [`pandas.DataFrame.std`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `PCA().fit(data)` garde tous les axes ; `pca.explained_variance_ratio_[0]` est la part de variance de PC1
- `pd.Series(pca.components_[0], index=X.columns)` : les coefficients de PC1, avec le nom de chaque mesure
- `.sort_values(key=abs, ascending=False)` les trie par valeur absolue décroissante
- `X.std()` : l'écart-type de chaque colonne, dans l'unité de la mesure
- attendu : sans réduction, PC1 garde 95 % de la variance, et son plus fort coefficient vaut 0,98 ; avec réduction, PC1 garde 28 %

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for name, data in [("non réduit", X), ("réduit", X_scaled)]:
  pca = ...
  coefficients = pd.Series(..., index=X.columns)
  print(f"{name} : PC1 garde {...:.0%} de la variance")
  print(coefficients.sort_values(key=abs, ascending=False).round(2).head(4))
  print()
print("Écart-type de chaque mesure :")
print(X.std().sort_values(...).round(4))
```

### Solution

In [ ]:
for name, data in [("non réduit", X), ("réduit", X_scaled)]:
  pca = PCA().fit(data)
  coefficients = pd.Series(pca.components_[0], index=X.columns)
  print(f"{name} : PC1 garde {pca.explained_variance_ratio_[0]:.0%} de la variance")
  print(coefficients.sort_values(key=abs, ascending=False).round(2).head(4))
  print()
print("Écart-type de chaque mesure :")
print(X.std().sort_values(ascending=False).round(4))
# Sans réduction, PC1 garde 95 % de la variance et c'est presque le dioxyde de soufre
# total seul (coefficient 0,98) : en mg/dm³, son écart-type (33) écrase ceux des
# autres mesures (moins de 2, sauf le soufre libre, 10). PC1 suit l'unité de la mesure,
# pas ce qu'elle dit des vins.
# Les 11 mesures ont des unités différentes (g/dm³, mg/dm³, % d'alcool, pH) : il faut
# réduire (ACP normée). PC1 ne garde plus que 28 % de la variance, et mêle plusieurs
# mesures (acidité fixe, acide citrique, pH, densité).

## 2. Choisir k

Comme sur la slide « Choisir k et passer en production » :

- ajustez sur `X_scaled` une `PCA()` qui garde les 11 axes, et tracez la variance expliquée cumulée en fonction de k ;
- affichez la part de la variance que gardent les 2 premiers axes, ceux d'un nuage de points ;
- trouvez le plus petit k qui garde 80 % et 95 % de la variance, et vérifiez que `PCA(0.8)` et `PCA(0.95)` trouvent les mêmes (leur attribut `n_components_`).

Que cache un nuage de points en 2D sur ce jeu ? Pourquoi faut-il tant d'axes, quand deux suffisent sur Iris (96 %) ?

**Documentation utile**

- [`numpy.cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html)
- [`numpy.searchsorted`](https://numpy.org/doc/stable/reference/generated/numpy.searchsorted.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.cumsum(pca.explained_variance_ratio_)` : la variance gardée par les k premiers axes, pour k = 1 à 11 ; `cumulative[1]` est celle des 2 premiers
- `np.searchsorted(cumulative, 0.95) + 1` : le plus petit k dont la variance cumulée atteint 95 % (`searchsorted` renvoie un indice, qui commence à 0)
- `plt.axhline(0.95, color="gray", linestyle="--")` repère le seuil sur le graphique
- attendu : 46 % pour 2 axes ; k = 6 pour 80 %, k = 9 pour 95 %

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
cumulative = np.cumsum(...)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Variance expliquée cumulée")
plt.show()

print(f"Les 2 premiers axes gardent : {...:.0%}")
for share in [0.8, 0.95]:
  k = ...
  n_components = ...
  print(f"{share:.0%} : k = {k}, PCA : {n_components}")
```

### Solution

In [ ]:
cumulative = np.cumsum(PCA().fit(X_scaled).explained_variance_ratio_)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Variance expliquée cumulée")
plt.show()

print(f"Les 2 premiers axes gardent : {cumulative[1]:.0%}")
for share in [0.8, 0.95]:
  k = np.searchsorted(cumulative, share) + 1
  n_components = PCA(share).fit(X_scaled).n_components_
  print(f"{share:.0%} : k = {k}, PCA : {n_components}")
# Les 2 premiers axes ne gardent que 46 % de la variance : un nuage en 2D montre moins
# de la moitié des différences entre les vins, et deux vins proches sur le dessin
# peuvent être loin dans les 11 mesures. Il faut 6 axes pour 80 %, 9 sur 11 pour 95 % :
# la courbe monte lentement, parce que les mesures sont peu corrélées entre elles et que
# la variance s'étale sur beaucoup de directions. Sur Iris, trois des quatre mesures
# sont très corrélées, et deux axes gardent 96 %.

## 3. Lire les axes : les vins extrêmes

- Ajustez une `PCA(n_components=2)` sur `X_scaled` et rangez la projection dans `X_pca` (les parties suivantes s'en servent). Affichez ses coefficients dans un `DataFrame` `components`, une ligne par axe et une colonne par mesure, et nommez chaque axe d'après ses plus forts coefficients, comme sur la slide « Lire une ACP ».
- Tracez le nuage de `X_pca`. Quelques vins se démarquent : trouvez celui qui a la plus grande coordonnée sur PC1, et celui qui a la plus grande sur PC2.
- Expliquez leur place. La coordonnée d'un vin sur un axe est la somme, sur les 11 mesures, de sa valeur réduite (sa ligne de `X_scaled`) fois le coefficient de la mesure : ce sont les contributions des mesures. Affichez les plus fortes pour chacun des deux vins, puis ses mesures dans `df`, à côté de leur médiane.

Quelles mesures placent chacun de ces vins si loin ? Est-ce le même genre de vin que le reste du jeu ?

**Documentation utile**

- [`numpy.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)
- [`pandas.DataFrame.iloc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pd.DataFrame(pca.components_, index=["PC1", "PC2"], columns=X.columns)` ; sa transposée, `components.T`, se lit mieux (une ligne par mesure)
- `np.argmax(X_pca[:, 0])` : la position du vin le plus à droite ; son numéro dans le CSV est `df.index[i]`
- les contributions du vin `i` à l'axe `k` : `X_scaled[i] * pca.components_[k]` ; leur somme vaut `X_pca[i, k]`
- `df.iloc[i]` : les mesures du vin à la position `i` ; `df.median()` : la médiane de chaque colonne
- attendu : le plus fort coefficient de PC1 est celui de l'acidité fixe (0,49 en valeur absolue) ; le vin le plus à droite est le n° 152 (PC1 = 8,7), le plus haut le n° 1 435 (PC2 = 6,8)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
pca = PCA(n_components=2)
X_pca = ...
components = pd.DataFrame(..., index=["PC1", "PC2"], columns=X.columns)
display(components.T.round(2))

plt.figure(figsize=(7, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=5, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

for k in range(2):
  i = ...
  contributions = pd.Series(..., index=X.columns)
  print(f"Vin n° {df.index[i]} : PC{k + 1} = {X_pca[i, k]:.1f}")
  print(contributions.sort_values(key=abs, ascending=False).round(2).head(4))
  display(pd.DataFrame({f"Vin n° {df.index[i]}": df.iloc[i], "médiane": df.median()}).T)
```

### Solution

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
components = pd.DataFrame(pca.components_, index=["PC1", "PC2"], columns=X.columns)
display(components.T.round(2))

plt.figure(figsize=(7, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=5, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

for k in range(2):
  i = np.argmax(X_pca[:, k])
  contributions = pd.Series(X_scaled[i] * pca.components_[k], index=X.columns)
  print(f"Vin n° {df.index[i]} : PC{k + 1} = {X_pca[i, k]:.1f}")
  print(contributions.sort_values(key=abs, ascending=False).round(2).head(4))
  display(pd.DataFrame({f"Vin n° {df.index[i]}": df.iloc[i], "médiane": df.median()}).T)
# PC1 oppose l'acidité (acidité fixe, acide citrique, densité) au pH : un axe « acidité ».
# PC2 oppose le soufre (dioxyde de soufre total et libre) à l'alcool.
# Le vin n° 152 doit sa place sur PC1 à ses chlorures (0,61 g/dm³, contre 0,08 pour la
# médiane), à ses sulfates (2,0 contre 0,62) et à son acide citrique, les plus forts du
# jeu, et à son pH très bas : un vin hors norme, pas un vin très acide comme les autres
# vins de droite. Le n° 1 435 doit sa place sur PC2 à son sucre résiduel (15,4 g/dm³,
# contre 2,2) et à son soufre libre : un vin presque doux, rare dans ce jeu (le
# n° 1 436, le même vin en double, est au même endroit). Un point extrême s'explique
# par les contributions : le coefficient d'une mesure ne suffit pas, il faut aussi que
# le vin s'écarte de la moyenne sur cette mesure.

## 4. La qualité se lit-elle sur la projection ?

- Affichez la répartition des notes de qualité.
- Rangez les vins en trois groupes dans une série `groups` : les mauvais (note inférieure à 5), les moyens (5 et 6) et les bons (7 et plus), et affichez l'effectif de chaque groupe.
- Complétez la fonction `show_projection(ax, Z, title)`, qui trace une projection 2D `Z` sur le graphique `ax`, chaque vin coloré par son groupe (les moyens, nombreux, en gris clair et transparents, sous les autres), puis tracez `X_pca` avec elle. La partie 5 la réutilise.

Les trois groupes se séparent-ils sur la projection ? Vers où penchent les bons vins, et quelle mesure de la partie 3 l'explique ? Pourquoi l'ACP, non supervisée, ne cherche-t-elle pas à séparer la qualité ?

**Documentation utile**

- [`pandas.Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
- [`pandas.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)

In [ ]:
def show_projection(ax, Z, title):
  # Trace la projection 2D Z sur ax, un point par vin, coloré par son groupe
  ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `df["quality"].value_counts().sort_index()` : le nombre de vins par note
- `pd.cut(df["quality"], bins=[0, 4, 6, 10], labels=[...])` : les intervalles sont fermés à droite, ]0, 4], ]4, 6] et ]6, 10]
- dans `show_projection`, une boucle sur les groupes, le gris en premier pour qu'il soit dessous : `mask = (groups == name).to_numpy()`, puis `ax.scatter(Z[mask, 0], Z[mask, 1], ...)`
- attendu : 63 mauvais, 1 319 moyens et 217 bons

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(df["quality"].value_counts().sort_index())
groups = pd.cut(..., bins=[0, 4, 6, 10], labels=["mauvais", "moyen", "bon"])
print(groups.value_counts())

COLORS = {"moyen": "lightgray", "mauvais": "tab:red", "bon": "tab:green"}


def show_projection(ax, Z, title):
  for name, color in COLORS.items():
    mask = ...
    alpha = 0.4 if name == "moyen" else 0.8
    ax.scatter(..., s=5, color=color, alpha=alpha, label=name)
  ax.set_title(title)
  ax.legend()


fig, ax = plt.subplots(figsize=(7, 6))
show_projection(ax, X_pca, "ACP")
plt.show()
```

### Solution

In [ ]:
print(df["quality"].value_counts().sort_index())
groups = pd.cut(df["quality"], bins=[0, 4, 6, 10], labels=["mauvais", "moyen", "bon"])
print(groups.value_counts())

COLORS = {"moyen": "lightgray", "mauvais": "tab:red", "bon": "tab:green"}


def show_projection(ax, Z, title):
  # Trace la projection 2D Z sur ax, un point par vin, coloré par son groupe
  for name, color in COLORS.items():
    mask = (groups == name).to_numpy()
    alpha = 0.4 if name == "moyen" else 0.8
    ax.scatter(Z[mask, 0], Z[mask, 1], s=5, color=color, alpha=alpha, label=name)
  ax.set_title(title)
  ax.legend()


fig, ax = plt.subplots(figsize=(7, 6))
show_projection(ax, X_pca, "ACP")
plt.show()
# Non : les trois groupes se superposent. Les bons vins penchent vers le bas de PC2,
# le côté de l'alcool (coefficient de -0,39 sur PC2, et l'alcool est la mesure la plus
# liée à la note) ; les mauvais un peu vers la gauche de PC1. Mais l'ACP ne connaît pas
# la note : elle cherche les directions de plus grande variance des 11 mesures
# (l'acidité, le soufre), qui ne sont pas celles qui font la qualité. L'axe de plus
# grande variance n'a aucune raison d'être le plus utile pour une cible : pour séparer
# des classes connues, il faut une méthode supervisée (la LDA, un modèle).

## 5. Projections non linéaires : UMAP et t-SNE

UMAP et t-SNE gardent les voisins de chaque vin plutôt que la variance. Comme le conseillent leurs bonnes pratiques, essayez-en plusieurs réglages, avec `random_state=0` pour retrouver la même carte à chaque exécution :

- projetez `X_scaled` en 2D avec UMAP pour 5, 15 et 50 voisins (`n_neighbors`), puis avec t-SNE pour une perplexité de 5, 30 et 50 ;
- tracez les six projections avec `show_projection`, sur une grille de 2 × 3 graphiques.

La qualité se sépare-t-elle mieux qu'avec l'ACP ? Qu'est-ce qui change avec le nombre de voisins ou la perplexité, et qu'est-ce qui reste d'une carte à l'autre ? Que peut-on lire dans la taille des groupes et les distances entre eux ?

**Documentation utile**

- [`umap.UMAP`](https://umap-learn.readthedocs.io/en/latest/api.html#umap.umap_.UMAP)
- [`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `umap.UMAP(n_neighbors=..., random_state=0, n_jobs=1)` et `TSNE(perplexity=..., random_state=0)` ont une méthode `fit_transform`, comme `PCA` ; avec `random_state`, UMAP calcule sur un seul cœur, et `n_jobs=1` évite qu'il le signale par un avertissement
- `fig, axes = plt.subplots(2, 3, figsize=(15, 9))` : `axes[0]` est la première ligne de graphiques, `axes[1]` la seconde
- attendu : quelques secondes par projection (la première de UMAP est plus longue : il compile son code)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import umap
from sklearn.manifold import TSNE

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, n_neighbors in zip(axes[0], [5, 15, 50]):
  reducer = umap.UMAP(...)
  show_projection(ax, ..., f"UMAP, n_neighbors = {n_neighbors}")
for ax, perplexity in zip(axes[1], [5, 30, 50]):
  reducer = TSNE(...)
  show_projection(ax, ..., f"t-SNE, perplexité = {perplexity}")
plt.show()
```

### Solution

In [ ]:
import umap
from sklearn.manifold import TSNE

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, n_neighbors in zip(axes[0], [5, 15, 50]):
  reducer = umap.UMAP(n_neighbors=n_neighbors, random_state=0, n_jobs=1)
  show_projection(
    ax, reducer.fit_transform(X_scaled), f"UMAP, n_neighbors = {n_neighbors}"
  )
for ax, perplexity in zip(axes[1], [5, 30, 50]):
  reducer = TSNE(perplexity=perplexity, random_state=0)
  show_projection(
    ax, reducer.fit_transform(X_scaled), f"t-SNE, perplexité = {perplexity}"
  )
plt.show()
# Pas vraiment : les bons vins se rassemblent un peu plus que sur l'ACP, d'un même côté
# de chaque carte, mais restent mêlés aux moyens. UMAP et t-SNE ne connaissent pas la
# note non plus : ils gardent les voisins de chaque vin dans les 11 mesures.
# Avec 5 voisins, UMAP émiette la carte en petits îlots et en points isolés (la
# structure très locale) ; avec 50, la forme d'ensemble ressort. t-SNE change moins,
# mais les bons vins s'y resserrent quand la perplexité monte. Ce qui reste d'une carte
# à l'autre, comme le côté des bons vins, est plus sûr que le détail des îlots.
# La taille des groupes et les distances entre eux ne sont pas fiables sur ces cartes.

## Conclusion

- Des mesures d'unités différentes se réduisent avant l'ACP : sinon, la plus dispersée fait PC1 à elle seule.
- La variance expliquée cumulée dit combien d'axes garder, et ce que cache un nuage en 2D (ici, plus de la moitié de la variance).
- Les coefficients nomment les axes ; les contributions expliquent la place d'un point.
- L'ACP ne cherche pas la cible : la qualité ne se lit pas sur ses premiers axes.
- UMAP et t-SNE : plusieurs réglages, `random_state` fixé, et ni la taille des groupes ni leurs distances ne se lisent.